In [ ]:
import pandas as pd
df = pd.read_csv( 'SAM_lion_results.csv')
print("working1")

In [ ]:
output_csv = "SAM_lion_results.csv"
df_sam_lion_pred = pd.read_csv(output_csv, usecols=['image_path', 'Species category'])

print("working2!")

In [ ]:
import json
# Load the JSON ground truth data
with open('/scratch/project_2001382/data/shared/zebra/annotations/GZCD_gt.json', 'r') as f:
    data = json.load(f)
    #print(f.read())

print("working3")

In [ ]:
df_sam_lion_gt = pd.DataFrame(data['annotations'])
#print("Ground truth table:", df_sam_zebra_gt)

df_sam_gt_cul = df_sam_lion_gt[['image_uuid', 'category', 'category_id']]


df_sam_lion_gt_id = pd.DataFrame(data['categories'])
df_sam_lion_gt_id_cul= df_sam_lion_gt_id[['id', 'name']]
#print(df_sam_plains_gt_id_cul)

# Keep only needed columns
#df_sam_zebra_gt_cul = df_sam_zebra_gt[['image_uuid', 'category']]
df_sam_lion_gt_cul = df_sam_lion_gt[['image_uuid', 'category', 'category_id']]
#df_sam_gt_id_cul = df_sam_plains_gt_cul[['category_id']]
#print(df_sam_gt_id_cul)
for i, row in df_sam_lion_gt_cul.iterrows():
    if row['category'] != 'lion':
        df_sam_lion_gt_cul.loc[i, 'category']= 'other'



output_csv = "SAM_lion_results.csv"
df_sam_lion_pred = pd.read_csv(output_csv, usecols=['image_path', 'Species category'])
#df_sam_zebra_pred = pd.read_csv(output_csv, usecols=['Species category'])
df_sam_lion_pred_cul = df_sam_lion_pred[['image_path', 'Species category']]
#print(df_sam_zebra_pred_cul)

#print("SAM results table:", df_sam_zebra_pred_cul).head(2)
#print(df_sam_pred.dtypes)
#print("Ground truth needed columns:", df_gt_cul)
#print(df_gt_cul.dtypes)

print("working3!")

In [ ]:
df_sam_lion_pred = df_sam_lion_pred_cul.drop([1925, 5893, 6768, 8011, 11578, 13022])
df_sam_lion_pred.reset_index(drop=True, inplace=True)
print(df_sam_lion_pred)#14597
print("working2!")

In [ ]:

#df_sam_lion_gt = pd.DataFrame(data['annotations'])
#df_sam_lion_gt_cul = df_sam_lion_gt[['image_uuid']]

#df_sam_zebra_pred_cul = df_sam_zebra_pred[['image_path', 'Species category']]
#df_sam_zebra_gt_cul = df_sam_zebra_gt[['image_uuid']]
#print(df_sam_zebra_pred_cul)
#print(df_sam_zebra_gt_cul)

In [ ]:
#update category labels and remove the index of empty file

from IPython.display import display

for i, row in df_sam_lion_gt_cul.iterrows():
    if row['category'] != 'lion':
        df_sam_lion_gt_cul.loc[i, 'category']= 'other'
#print (df_sam_zebra_gt_cul)

df_info_gt= df_sam_lion_gt_cul[['image_uuid', 'category', 'category_id']]
#pd.set_option('display.max_rows', None)
#df_info_gt = pd.DataFrame(df_sam_zebra_gt_cul)
#print(df_info_gt)

#drop empty images_uuid, category and category_id by their index:
df_new_cul_gt =df_info_gt.drop([1040, 1041, 1042, 1043, 1044, 1045, 1046, 11600, 11958, 11959, 11960, 11961, 12232, 875, 876, 877, 2274, 2940, 4420, 4475, 4476,4477, 4478, 4479, 4649, 4650, 5370, 5371, 5372, 6573, 6873,6874, 6875, 8491, 8492, 8493, 8494, 8495, 8618, 9985])
df_new_cul_gt.reset_index(drop=True, inplace=True)
#display(df_new_cul_gt)
print(df_new_cul_gt)

In [ ]:
#new2
#remove the extension .jpg from image_path
import os
import cv2
from PIL import Image
import matplotlib.pyplot as plt


names = []
for i, row in df_sam_lion_pred_cul.iterrows():
    image_name = row['image_path']
    full_path = os.path.basename(image_name)
    #uudi = os.path.splitext(full_path)[0]
    uudi = full_path.split("_")[0] #remove everything after "_"
    names.append(uudi)
#print(names)

df_names =pd.DataFrame(names, columns=['image_uuid'])
df_id_category_cul = pd.concat([df_names, df_sam_lion_pred_cul['Species category']], axis=1)
#print(df_id_category_cul)

In [ ]:
#filter matched images with bounding boxes and category:
#df_matched_images

char_bar_merge= pd.merge(df_sam_gt_cul, df_id_category_cul, how="inner", on="image_uuid")

#Matched_details= df_names.merge(df_new_cul_gt [['image_uuid', 'category_id', 'category']] , how="inner", on="image_uuid")
#Matched_details= pd.merge(df_id_category_cul, df_new_cul_gt, how="inner", on="image_uuid")
eka=df_id_category_cul['_idx'] = df_id_category_cul.groupby('image_uuid').cumcount()
toka=df_new_cul_gt['_idx'] = df_new_cul_gt.groupby('image_uuid').cumcount()
#print(eka)
#print(toka)
df_merged = pd.merge(df_id_category_cul, df_new_cul_gt, on=['image_uuid', '_idx'])
print(df_merged)
#Matched_details= df_matched_images.merge(df_sam_zebra_gt_category [['image_uuid', 'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h', 'category']] , how="inner", on="image_uuid")
#print("Matched images with bboxes: ", Matched_details)  
#display(Matched_details.head(20))
print("working4!")

In [ ]:
#Confusion matrix for SAM lion

from PIL import Image
import pandas as pd
import glob
import json
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, balanced_accuracy_score
from sklearn.metrics import ConfusionMatrixDisplay

actual = df_merged['category']    # ground truth
predicted = df_merged['Species category']

labels = ['lion', 'other']
# Confusion matrix, accuracy, and full classification report
cm = confusion_matrix(actual, predicted, labels=labels)
#print("Confusion matrix:", cm)
#cm_display = ConfusionMatrixDisplay(confusion_matrix = cm, display_labels =labels)
#cm_display.plot()

# xticklabels=['Predicted Positive', 'Predicted Negative'],
 #yticklabels=['Actual Positive', 'Actual Negative'])

plt.figure(figsize=(5,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens",
 xticklabels=labels,
 yticklabels=labels)
plt.xlabel('SAM v3 Categorization ')
plt.ylabel('Ground Truth')
plt.title('Confusion matrix')
#plt.savefig('cm.jpg')
plt.show()

In [ ]:
accuracy = accuracy_score(actual, predicted)
print("Accuracy:", accuracy)
print(classification_report(actual, predicted))
Bal_accuracy =balanced_accuracy_score (actual, predicted)
print("Balenced Accuracy:", Bal_accuracy)

print("working5!")

In [ ]:
import os
import cv2
from PIL import Image
import matplotlib.pyplot as plt


condition1 = char_bar_merge.loc[(char_bar_merge['Species category'] =='lion') & (char_bar_merge['category'] != 'lion')]
condition1['category'].value_counts().plot(kind='barh')
plt.title('SAM v3 miscategorized species')
